## Trial submission of LightGBM with no tuning
* from `window_hyperparam.ipynb`: Choose an Expanding window with recency_decay of 0.03
* from `soft_label_weight_tune`: The confidence cut off is 0.6, and soft label weight is 0.5
-----
Here is a trial submission using an un-tuned LightGBM model

In [1]:
import numpy as np
import pandas as pd

from sklearn.utils.class_weight import compute_sample_weight
from lightgbm import LGBMClassifier

In [2]:
train_raw = pd.read_csv("train.csv")
train_soft_label_raw = pd.read_csv("train_soft_label.csv")
test_raw = pd.read_csv("test.csv")
edges_raw = pd.read_csv("txs_edgelist.csv")


print("Train:", train_raw.shape)
print("Soft-label train:", train_soft_label_raw.shape)
print("Test:", test_raw.shape)
print("Soft-label columns:", train_soft_label_raw.columns.tolist())

Train: (141772, 168)
Soft-label train: (141772, 170)
Test: (15329, 168)
Soft-label columns: ['txId', 'time_step', 'feat_1', 'feat_2', 'feat_3', 'feat_4', 'feat_5', 'feat_6', 'feat_7', 'feat_8', 'feat_9', 'feat_10', 'feat_11', 'feat_12', 'feat_13', 'feat_14', 'feat_15', 'feat_16', 'feat_17', 'feat_18', 'feat_19', 'feat_20', 'feat_21', 'feat_22', 'feat_23', 'feat_24', 'feat_25', 'feat_26', 'feat_27', 'feat_28', 'feat_29', 'feat_30', 'feat_31', 'feat_32', 'feat_33', 'feat_34', 'feat_35', 'feat_36', 'feat_37', 'feat_38', 'feat_39', 'feat_40', 'feat_41', 'feat_42', 'feat_43', 'feat_44', 'feat_45', 'feat_46', 'feat_47', 'feat_48', 'feat_49', 'feat_50', 'feat_51', 'feat_52', 'feat_53', 'feat_54', 'feat_55', 'feat_56', 'feat_57', 'feat_58', 'feat_59', 'feat_60', 'feat_61', 'feat_62', 'feat_63', 'feat_64', 'feat_65', 'feat_66', 'feat_67', 'feat_68', 'feat_69', 'feat_70', 'feat_71', 'feat_72', 'feat_73', 'feat_74', 'feat_75', 'feat_76', 'feat_77', 'feat_78', 'feat_79', 'feat_80', 'feat_81', 'fea

With the selected parameters

In [ ]:
SOFT_LABEL_WEIGHT = 0.50
FUTURE_LABEL_WEIGHT = 0.8
CONFIDENCE_CUTOFF = 0.60
RECENCY_DECAY = 0.03
INCLUDE_TIME_STEP = False

soft_label_col = "p_illicit"

assert soft_label_col in train_soft_label_raw.columns
assert "label" in train_raw.columns
assert "txId" in train_raw.columns
assert "txId" in test_raw.columns

In [4]:
edges = edges_raw[
    ["txId1", "txId2"]
].drop_duplicates(
    subset=["txId1", "txId2"]
).copy()

edges = edges[
    edges["txId1"] != edges["txId2"]
].copy()

in_degree = edges["txId2"].value_counts()
out_degree = edges["txId1"].value_counts()

node_features = pd.DataFrame({
    "txId": pd.concat(
        [train_raw["txId"], test_raw["txId"]],
        ignore_index=True,
    ).drop_duplicates()
})

node_features["in_degree"] = (
    node_features["txId"].map(in_degree).fillna(0).astype("float32")
)
node_features["out_degree"] = (
    node_features["txId"].map(out_degree).fillna(0).astype("float32")
)
node_features["total_degree"] = (
    node_features["in_degree"] + node_features["out_degree"]
)
node_features["degree_ratio"] = (
    (node_features["in_degree"] + 1.0)
    / (node_features["out_degree"] + 1.0)
)

In [5]:
# Calculate transaction volume using both train and test rows,
# since test periods are known at prediction time.
all_transaction_times = pd.concat(
    [
        train_raw[["txId", "time_step"]],
        test_raw[["txId", "time_step"]],
    ],
    ignore_index=True,
).drop_duplicates("txId")

step_volume = all_transaction_times["time_step"].value_counts()

node_features = node_features.merge(
    all_transaction_times,
    on="txId",
    how="left",
)

node_features["timestep_tx_volume"] = (
    node_features["time_step"]
    .map(step_volume)
    .astype("float32")
)

node_features = node_features.drop(
    columns=["time_step"]
)

graph_feature_cols = [
    "in_degree",
    "out_degree",
    "total_degree",
    "degree_ratio",
    "timestep_tx_volume",
]

display(node_features.head())

,txId,in_degree,out_degree,total_degree,degree_ratio,timestep_tx_volume
0,230425980,1.0,1.0,2.0,1.000000,7880.0
1,5530458,1.0,1.0,2.0,1.000000,7880.0
2,232022460,1.0,2.0,3.0,0.666667,7880.0
3,232438397,160.0,1.0,161.0,80.500000,7880.0
4,230460314,2.0,8.0,10.0,0.333333,7880.0


In [6]:
train_final = train_raw.merge(
    node_features,
    on="txId",
    how="left",
)

test_final = test_raw.merge(
    node_features,
    on="txId",
    how="left",
)

# Add the same full-graph features to the soft-label table.
soft_final = train_soft_label_raw.merge(
    node_features,
    on="txId",
    how="left",
)

for frame in [train_final, test_final, soft_final]:
    frame[graph_feature_cols] = (
        frame[graph_feature_cols].fillna(0)
    )

print("Train with graph features:", train_final.shape)
print("Test with graph features:", test_final.shape)
print("Soft data with graph features:", soft_final.shape)

Train with graph features: (141772, 173)
Test with graph features: (15329, 173)
Soft data with graph features: (141772, 175)


In [7]:
confirmed = train_final[
    train_final["label"].notna()
].copy()

confirmed["training_label"] = (
    confirmed["label"].astype(int)
)
confirmed["label_source_weight"] = 1.0

pseudo = soft_final[
    soft_final["label"].isna()
    & soft_final[soft_label_col].notna()
].copy()

# Cutoff 0.60 means p <= 0.40 becomes licit and p >= 0.60
# becomes illicit. Values strictly between these are excluded.
pseudo = pseudo[
    (pseudo[soft_label_col] <= 1.0 - CONFIDENCE_CUTOFF)
    | (pseudo[soft_label_col] >= CONFIDENCE_CUTOFF)
].copy()

pseudo["training_label"] = (
    pseudo[soft_label_col] >= 0.5
).astype(int)
pseudo["label_source_weight"] = SOFT_LABEL_WEIGHT

selected_columns = list(dict.fromkeys(
    [
        "txId",
        "time_step",
        "training_label",
        "label_source_weight",
    ]
    + [
        c for c in train_final.columns
        if c not in ["txId", "label", "training_label", "label_source_weight"]
    ]
))

final_train = pd.concat(
    [
        confirmed[selected_columns],
        pseudo[selected_columns],
    ],
    ignore_index=True,
)

final_train = final_train.loc[
    :, ~final_train.columns.duplicated()
]

print("Confirmed rows:", len(confirmed))
print("Pseudo-labelled rows:", len(pseudo))
print("Final training rows:", len(final_train))
print(final_train["training_label"].value_counts())

Confirmed rows: 31235
Pseudo-labelled rows: 108410
Final training rows: 139645
training_label
0    130040
1      9605
Name: count, dtype: int64


In [ ]:
def create_lightgbm():
    return LGBMClassifier(
        objective="binary",

        random_state=100,
        n_jobs=-1,

    )

# Feature columns
original_feature_cols = [
    c for c in train_raw.columns
    if c not in ["txId", "label"]
]

final_feature_cols = list(dict.fromkeys(
    original_feature_cols + graph_feature_cols
))

if not INCLUDE_TIME_STEP:
    final_feature_cols.remove("time_step")

# Historical training data: steps 1–35
history = final_train.copy()

# Ensure the columns required for future pseudo-labelled rows exist
history["training_label"] = history["training_label"].astype(int)
history["label_source_weight"] = history["label_source_weight"].astype(float)

test_steps = sorted(test_final["time_step"].unique())

predictions = []

for current_step in test_steps:

    print(f"Predicting time_step {current_step}")

    # Train only on all steps before the current prediction step
    train_history = history[
        history["time_step"] < current_step
    ].copy()

    X_history = train_history[final_feature_cols]
    y_history = train_history["training_label"].astype(int)

    # Recalculate class weights using the current expanding dataset
    class_weights = compute_sample_weight(
        class_weight="balanced",
        y=y_history,
    )

    # The latest training step is immediately before current_step
    training_cutoff = current_step - 1

    age = (
        training_cutoff
        - train_history["time_step"].to_numpy()
    )

    recency_weights = np.exp(
        -RECENCY_DECAY * age
    )

    source_weights = (
        train_history["label_source_weight"].to_numpy()
    )

    sample_weights = (
        class_weights
        * recency_weights
        * source_weights
    )

    # Normalise weights for numerical stability
    sample_weights = (
        sample_weights / sample_weights.mean()
    )

    # Fit a new model on the expanding historical window
    model = create_lightgbm()

    model.fit(
        X_history,
        y_history,
        sample_weight=sample_weights,
    )

    # Predict only the current test time step
    current_test = test_final[
        test_final["time_step"] == current_step
    ].copy()

    X_current = current_test[final_feature_cols]

    current_probabilities = model.predict_proba(
        X_current
    )[:, 1]

    current_test["target"] = current_probabilities

    predictions.append(
        current_test[["index", "txId", "time_step", "target"]]
    )

    # Add the newly predicted step to the expanding training data
    # so that it is available when predicting the next time step.
    newly_labelled = current_test.copy()

    newly_labelled["training_label"] = (
        newly_labelled["target"] >= 0.5
    ).astype(int)

    newly_labelled["label_source_weight"] = (
        FUTURE_LABEL_WEIGHT
    )

    # Remove columns that are not needed by the next model
    newly_labelled = newly_labelled[
        history.columns
    ]

    history = pd.concat(
        [history, newly_labelled],
        ignore_index=True,
    )

    print(
        f"Added {len(newly_labelled):,} predicted rows "
        f"from step {current_step} to training history."
    )

Predicting time_step 36
[LightGBM] [Info] Number of positive: 9605, number of negative: 130040
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.145461 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 39615
[LightGBM] [Info] Number of data points in the train set: 139645, number of used features: 170
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.563020 -> initscore=0.253428
[LightGBM] [Info] Start training from score 0.253428
Added 1,708 predicted rows from step 36 to training history.
Predicting time_step 37
[LightGBM] [Info] Number of positive: 9641, number of negative: 131712
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.127934 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 39649
[LightGBM] [Info] Number of data points in the train set: 141353, number of used features: 170
[LightGBM] [Info] [binary:BoostFr

In [9]:
prediction_results = pd.concat(
    predictions,
    ignore_index=True,
)

submission = pd.DataFrame({
    "index": prediction_results["index"],
    "target": prediction_results["target"],
})

assert len(submission) == len(test_raw)
assert submission["target"].between(0, 1).all()

# Confirm that test order has been preserved
assert submission["index"].tolist() == test_final["index"].tolist()

display(submission.head())
print("Submission shape:", submission.shape)
display(submission["target"].describe())

,index,target
0,0,0.002143
1,1,0.008649
2,2,0.009164
3,3,0.003073
4,4,0.000486


Submission shape: (15329, 2)


count    15329.000000
mean         0.051263
std          0.197608
min          0.000064
25%          0.000520
50%          0.001329
75%          0.006449
max          0.999597
Name: target, dtype: float64

In [10]:
submission_path = (
    "submission_lgbm_expanding_soft_labels.csv"
)

submission.to_csv(
    submission_path,
    index=False,
)

print(f"Saved submission to: {submission_path}")

Saved submission to: submission_lgbm_expanding_soft_labels.csv
